# Carga de CSV

## Importar Librerías

In [1]:
import yfinance as yf
import pandas as pd
import numpy as np
import math
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, SimpleRNN, LSTM, GRU, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

## Descarga de Datos

In [2]:
# Definir los símbolos de las acciones
bbva_ticker = "BBVA.MC"   # BBVA (Bolsa de Madrid)
santander_ticker = "SAN.MC"  # Banco Santander (Bolsa de Madrid)

# Definir el rango de fechas
start_date = "2000-01-01"
end_date   = "2025-11-01"

# Descargar los datos históricos desde Yahoo Finance
bbva_data = yf.download(bbva_ticker, start=start_date, end=end_date, auto_adjust=False, actions=True)
santander_data = yf.download(santander_ticker, start=start_date, end=end_date, auto_adjust=False, actions=True)

# Mostrar resumen por consola
print("BBVA data:")
print(bbva_data.head())
print("\nSantander data:")
print(santander_data.head())

# Guardar los datos en CSV
bbva_data.to_csv("../csv/bbva_data.csv")
santander_data.to_csv("../csv/santander_data.csv")

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed

BBVA data:
Price      Adj Close      Close Dividends       High        Low       Open  \
Ticker       BBVA.MC    BBVA.MC   BBVA.MC    BBVA.MC    BBVA.MC    BBVA.MC   
Date                                                                         
2000-01-03  4.040033  13.623349       0.0  13.757854  13.594527  13.690602   
2000-01-04  3.934618  13.267874       0.0  13.536882  13.219837  13.450416   
2000-01-05  3.846295  12.970044       0.0  13.210230  12.912399  13.142977   
2000-01-06  3.846295  12.970044       0.0  12.970044  12.970044  12.970044   
2000-01-07  3.894730  13.133370       0.0  13.248659  12.998866  13.248659   

Price      Stock Splits    Volume  
Ticker          BBVA.MC   BBVA.MC  
Date                               
2000-01-03          0.0   8244257  
2000-01-04          0.0   8522096  
2000-01-05          0.0  12159826  
2000-01-06          0.0         0  
2000-01-07          0.0  62261944  

Santander data:
Price      Adj Close     Close Dividends      High       Lo

In [3]:
import yfinance as yf
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
import numpy as np

# ===========================
# Tickers relacionados
# ===========================
santander_ticker = "SAN.MC"
bbva_ticker = "BBVA.MC"
ibex_ticker = "^IBEX"
eurusd_ticker = "EURUSD=X"
sp500_ticker = "^GSPC"
oil_ticker = "CL=F"

start_date = "2000-01-01"
end_date   = "2025-11-01"

# ===========================
# Descargar datos con columna 'Close'
# ===========================
data_san   = yf.download(santander_ticker, start=start_date, end=end_date, auto_adjust=False, actions=True)[["Adj Close", "Close", "Volume", "High", "Low", "Open"]]
data_bbva  = yf.download(bbva_ticker, start=start_date, end=end_date, auto_adjust=False, actions=True)[["Close"]]
data_ibex  = yf.download(ibex_ticker, start=start_date, end=end_date, auto_adjust=False, actions=True)[["Close"]]
data_sp500 = yf.download(sp500_ticker, start=start_date, end=end_date, auto_adjust=False, actions=True)[["Close"]]
data_eurusd = yf.download(eurusd_ticker, start=start_date, end=end_date, auto_adjust=False, actions=True)[["Close"]]
data_oil   = yf.download(oil_ticker, start=start_date, end=end_date, auto_adjust=False, actions=True)[["Close"]]

# ===========================
# Renombrar columnas
# ===========================
data_san.rename(columns={"Adj Close": "SAN_Adj_Close", "Close": "SAN_Close", "Volume": "SAN_Volume", "High": "SAN_High", "Low": "SAN_Low", "Open": "SAN_Open"}, inplace=True)
data_bbva.rename(columns={"Close": "BBVA_Price"}, inplace=True)
data_ibex.rename(columns={"Close": "IBEX"}, inplace=True)
data_sp500.rename(columns={"Close": "SP500"}, inplace=True)
data_eurusd.rename(columns={"Close": "EURUSD"}, inplace=True)
data_oil.rename(columns={"Close": "OIL"}, inplace=True)

# ===========================
# Unir todo por fecha
# ===========================
df_full = data_san.join([data_bbva, data_ibex, data_sp500, data_eurusd, data_oil], how="inner")
df_full.dropna(inplace=True)

print("✅ Dataset multivariable creado con columnas:")
print(df_full.columns.tolist())


# ===========================
# Guardar CSV normalizado
# ===========================
df_full.to_csv("../csv/santander_enriched.csv")
print("💾 Archivo guardado: 'csv/santander_enriched_scaled.csv'")

# Mostrar ejemplo
print(df_full.head())


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed

✅ Dataset multivariable creado con columnas:
[('SAN_Adj_Close', 'SAN.MC'), ('SAN_Close', 'SAN.MC'), ('SAN_Volume', 'SAN.MC'), ('SAN_High', 'SAN.MC'), ('SAN_Low', 'SAN.MC'), ('SAN_Open', 'SAN.MC'), ('BBVA_Price', 'BBVA.MC'), ('IBEX', '^IBEX'), ('SP500', '^GSPC'), ('EURUSD', 'EURUSD=X'), ('OIL', 'CL=F')]
💾 Archivo guardado: 'csv/santander_enriched_scaled.csv'
Price      SAN_Adj_Close SAN_Close SAN_Volume  SAN_High   SAN_Low  SAN_Open  \
Ticker            SAN.MC    SAN.MC     SAN.MC    SAN.MC    SAN.MC    SAN.MC   
Date                                                                          
2003-12-01      2.415460  7.746234   42034006  7.772199  7.538513  7.746234   
2003-12-02      2.404664  7.711614  187525101  7.763544  7.633719  7.711614   
2003-12-03      2.420856  7.763544   31459710  7.763544  7.651029  7.763544   
2003-12-04      2.410062  7.728924   37904447  7.746234  7.668339  7.728924   
2003-12-05      2.401965  7.702959   20656590  7.728924  7.651029  7.702959   

Price  

In [4]:
# leer sin cabecera porque las tres primeras filas son especiales
df_raw = pd.read_csv("../csv/santander_enriched.csv", header=None)

# 1) la primera fila tiene los nombres buenos
column_names = df_raw.iloc[0].tolist()

# 2) nos quedamos con los datos reales (a partir de la fila 3 → índice 3)
df = df_raw.iloc[3:].reset_index(drop=True)

# 3) ponemos los nombres de columna
df.columns = column_names

# 4) renombrar 'Price' -> 'Date' (porque es la fecha en realidad)
df = df.rename(columns={"Price": "Date"})

# 5) convertir Date a datetime
df["Date"] = pd.to_datetime(df["Date"])

# 6) pasar el resto a numérico
for col in df.columns:
    if col != "Date":
        df[col] = pd.to_numeric(df[col], errors="coerce")

print(df.head())
print(df.dtypes)

df.to_csv("../csv/santander_enriched.csv")
print("💾 Archivo guardado: 'csv/santander_enriched.csv'")


        Date  SAN_Adj_Close  SAN_Close  SAN_Volume  SAN_High   SAN_Low  \
0 2003-12-01       2.415460   7.746234    42034006  7.772199  7.538513   
1 2003-12-02       2.404664   7.711614   187525101  7.763544  7.633719   
2 2003-12-03       2.420856   7.763544    31459710  7.763544  7.651029   
3 2003-12-04       2.410062   7.728924    37904447  7.746234  7.668339   
4 2003-12-05       2.401965   7.702959    20656590  7.728924  7.651029   

   SAN_Open  BBVA_Price         IBEX        SP500    EURUSD        OIL  
0  7.746234    9.895663  7372.299805  1070.119995  1.196501  29.950001  
1  7.711614    9.857233  7348.700195  1066.619995  1.208897  30.780001  
2  7.763544    9.924485  7384.299805  1064.729980  1.212298  31.100000  
3  7.728924    9.953307  7367.100098  1069.719971  1.208094  31.260000  
4  7.702959    9.857233  7348.100098  1061.500000  1.218695  30.730000  
Date             datetime64[ns]
SAN_Adj_Close           float64
SAN_Close               float64
SAN_Volume           

In [5]:
import pandas as pd

# Cargar el CSV
df = pd.read_csv("../csv/santander_enriched.csv", parse_dates=["Date"])
df.sort_values("Date", inplace=True)

# Crear la columna binaria
df["eventos_negativos"] = 0

# Lista definitiva de eventos negativos que afectan a España
eventos_negativos = [
    ("2000-03-01", "2002-12-31"),  # Burbuja puntocom
    ("2004-03-11", "2004-03-31"),  # Atentados 11M
    ("2008-09-01", "2009-06-30"),  # Crisis financiera global
    ("2010-05-01", "2012-12-31"),  # Crisis deuda europea
    ("2012-06-01", "2013-06-30"),  # Rescate bancario español
    ("2020-02-15", "2021-06-30"),  # COVID-19
    ("2022-02-24", "2023-12-31"),  # Guerra Rusia-Ucrania
]

# Marcar los periodos en la columna
for inicio, fin in eventos_negativos:
    mask = (df["Date"] >= inicio) & (df["Date"] <= fin)
    df.loc[mask, "eventos_negativos"] = 1

# Guardar el CSV actualizado
df.to_csv("../csv/santander_enriched.csv", index=False)

print("✅ Columna 'eventos_negativos' añadida correctamente.")


✅ Columna 'eventos_negativos' añadida correctamente.


In [9]:
import pandas as pd
import requests

url = "https://api.worldbank.org/v2/country/ESP/indicator/NY.GDP.MKTP.KD.ZG"
params = {"format": "json", "per_page": 2000}
data = requests.get(url, params=params).json()

rows = []
for item in data[1]:
    rows.append({"date": item["date"], "gdp_growth": item["value"]})

df_gdp = pd.DataFrame(rows)
df_gdp["date"] = pd.to_datetime(df_gdp["date"])  # anual
